In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import jax
import jax.numpy as jnp
from jax import vmap
import numpy as np
import matplotlib.pyplot as plt

from ml_collections import config_flags
# from configs.baseline import get_config
from configs.fixed_pseudo_time import get_config

from jaxpi.models import create_lr_schedule, create_optimizer, create_arch, create_train_state
from jaxpi.checkpointing import create_checkpoint_manager, restore_checkpoint

import models
from utils import get_dataset

In [ ]:
config = get_config()

rho_ref, u_ref, p_ref, T, X, t_star, x_star, left_coords, right_coords = get_dataset()

rho0 = rho_ref[:, 0]
u0   = u_ref[:, 0]
p0   = p_ref[:, 0]

print(f"t: [{t_star[0]:.4f}, {t_star[-1]:.4f}]  ({len(t_star)} pts)")
print(f"x: [{x_star[0]:.4f}, {x_star[-1]:.4f}]  ({len(x_star)} pts)")
print(f"rho_ref shape: {rho_ref.shape}  (Nx, Nt)")

In [ ]:
CKPT_DIR = os.path.join(os.getcwd(), config.wandb.name, "ckpt")
print("Looking for checkpoints in:", CKPT_DIR)

lr    = create_lr_schedule(config.optim)
tx    = create_optimizer(config.optim, lr)
arch  = create_arch(config.arch)
state = create_train_state(config, tx, arch)

model = models.Euler1D(
    config, lr, tx, arch, state,
    rho0, u0, p0, t_star, x_star, left_coords, right_coords
)

ckpt_mngr = create_checkpoint_manager(config.saving, CKPT_DIR)
print("Latest checkpoint step:", ckpt_mngr.latest_step())

model.state = restore_checkpoint(ckpt_mngr, model.state)
print("Restored at step:", int(model.state.step))

## Run predictions on the full grid

In [ ]:
rho_pred, u_pred, p_pred = vmap(
    vmap(model.neural_net, (None, 0, None)), (None, None, 0)
)(model.state.params, t_star, x_star)

print("Prediction shapes:", rho_pred.shape, u_pred.shape, p_pred.shape)

## L2 relative errors

In [ ]:
def l2_error(pred, ref):
    return float(jnp.linalg.norm(pred - ref) / jnp.linalg.norm(ref))

print(f"L2 error  rho : {l2_error(rho_pred, rho_ref):.4e}")
print(f"L2 error  u   : {l2_error(u_pred,   u_ref):.4e}")
print(f"L2 error  p   : {l2_error(p_pred,   p_ref):.4e}")

## Contour plots: predicted vs reference

In [ ]:
fields = [
    (rho_pred, rho_ref, r"$\rho$"),
    (u_pred,   u_ref,   r"$u$"),
    (p_pred,   p_ref,   r"$p$"),
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for row, (pred, ref, label) in enumerate(fields):
    pred_np = np.array(pred)
    ref_np  = np.array(ref)
    err_np  = np.abs(pred_np - ref_np)
    vmin, vmax = ref_np.min(), ref_np.max()

    for col, (data, title) in enumerate([
        (ref_np,  f"{label} Reference"),
        (pred_np, f"{label} Predicted"),
        (err_np,  f"{label} Abs. Error"),
    ]):
        ax = axes[row, col]
        if col < 2:
            cf = ax.contourf(T, X, data, levels=200, cmap="jet", vmin=vmin, vmax=vmax)
        else:
            cf = ax.contourf(T, X, data, levels=200, cmap="jet")
        fig.colorbar(cf, ax=ax)
        ax.set_title(title)
        ax.set_xlabel("$t$")
        ax.set_ylabel("$x$")

plt.tight_layout()
plt.show()

## Slice plots at fixed time snapshots

In [ ]:
t_indices = [0, len(t_star)//4, len(t_star)//2, len(t_star)-1]

fig, axes = plt.subplots(len(t_indices), 3, figsize=(15, 4 * len(t_indices)))

for row, ti in enumerate(t_indices):
    t_val = float(t_star[ti])
    for col, (pred, ref, label) in enumerate(fields):
        ax = axes[row, col]
        ax.plot(x_star, np.array(ref[:, ti]),  "k-",  lw=2,   label="Reference")
        ax.plot(x_star, np.array(pred[:, ti]), "r--", lw=1.5, label="Predicted")
        ax.set_title(f"{label}  at  t = {t_val:.4f}")
        ax.set_xlabel("$x$")
        if col == 0:
            ax.legend()

plt.tight_layout()
plt.show()